In [ ]:
pwd

'd:\\LAB\\ADR_DRAFT\\DRAFT\\HumanEval'

## Author Study

In [ ]:
# Load the JSONL file and sample 64 primary keys for later use.
import json
import random
from pathlib import Path

# Change this to any JSONL file you want to use
jsonl_path = Path("../DRAFT/Results/gemini-2.5-flash_TBtest.jsonl")

with jsonl_path.open("r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

primary_keys = [row["PrimaryKey"] for row in rows]

random.seed(42)  # makes the sample reproducible
sampled_keys = random.sample(primary_keys, 64)

print("Total rows:", len(primary_keys))
print("Sampled 64 primary keys:")
print(sampled_keys)

Total rows: 870
Sampled 64 primary keys:
[3849, 1353, 3382, 4223, 3846, 51, 1684, 2896, 250, 2875, 3325, 1610, 844, 3013, 3159, 740, 3596, 2804, 3231, 121, 3247, 3926, 1069, 560, 3056, 3341, 3508, 1952, 429, 501, 1275, 4168, 1482, 1876, 1213, 2016, 3379, 3397, 1985, 949, 3284, 2889, 709, 3096, 982, 3344, 1375, 993, 3812, 1614, 855, 4234, 3286, 3652, 3824, 2911, 2062, 3143, 4075, 66, 2544, 3881, 986, 945]


In [ ]:
# Save the sampled 64 primary keys to a JSON file for reuse.
import json
from pathlib import Path

# Save the sampled primary keys as a JSON file.
output_json_path = Path.cwd() / "sampled_keys.json"

payload = {"author": sampled_keys}
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Saved sampled keys to {output_json_path}")

Saved sampled keys to d:\LAB\ADR_DRAFT\DRAFT\HumanEval\sampled_keys.json


In [ ]:
# Build the CD task output table for the sampled keys and write it to a JSONL file.
import json
from pathlib import Path

# Use the sampled primary keys already produced in the previous cell.
# These are the 64 keys we want to build records for.
if 'sampled_keys' not in globals():
    raise NameError("Run the previous cell first so 'sampled_keys' is defined.")

# Locate the project root from the current notebook working directory.
base_dir = Path.cwd()
if not (base_dir / "HumanEval").exists():
    base_dir = next(
        (p for p in [base_dir, *base_dir.parents] if (p / "HumanEval").exists()),
        base_dir,
    )

# Paths to the four CD result files, one per approach.
files = {
    "Prompting": base_dir / "Prompting" / "Results" / "gemini-2.5-flash_CDtest.jsonl",
    "Finetuning": base_dir / "Finetune" / "Results" / "qwen3-30b-a3b-instruct-CDtest.jsonl",
    "RAFG": base_dir / "RAFG" / "Results" / "Qwen3-30B-A3B-Instruct-2507-CDtest-results.jsonl",
    "DRAFT": base_dir / "DRAFT" / "Results" / "qwen3-30b-a3b-instruct-CDtest.jsonl",
}

# Path to the ADR source data for the CD task.
source_path = base_dir / "Data" / "ADR-data" / "test.jsonl"


def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def get_primary_key(row):
    return row.get("PrimaryKey") or row.get("primary_key") or row.get("primaryKey")


def get_cd_value(row):
    # CD approach files hold the generated answer in Decision, while some files may use Body.
    return row.get("Decision") or row.get("decision")

# Load all rows from the four CD approaches.
approach_rows = {name: load_jsonl(path) for name, path in files.items()}

# Create a lookup from primary key to row for each approach.
lookups = {
    name: {get_primary_key(row): row for row in rows if get_primary_key(row) is not None}
    for name, rows in approach_rows.items()
}

# Load the ADR source rows and create a lookup by primary key.
source_rows = load_jsonl(source_path)
source_lookup = {
    get_primary_key(row): row
    for row in source_rows
    if get_primary_key(row) is not None
}

# Build one output dict per sampled primary key for the CD task.
output_rows = []
for pk in sampled_keys:
    source_row = source_lookup.get(pk)
    row_dict = {
        "PrimaryKey": pk,
        "Context": source_row.get("Context") if source_row else None,
        "Decision": source_row.get("Decision") if source_row else None,
    }
    for name in ["Prompting", "Finetuning", "RAFG", "DRAFT"]:
        row = lookups[name].get(pk)
        row_dict[name] = get_cd_value(row) if row is not None else None
    output_rows.append(row_dict)

# Save the combined records as JSONL in the HumanEval folder.
output_path = base_dir / "HumanEval" / "cd_64_samples.jsonl"
with output_path.open("w", encoding="utf-8") as f:
    for item in output_rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Wrote {len(output_rows)} records to {output_path}")

Wrote 64 records to d:\LAB\ADR_DRAFT\DRAFT\HumanEval\cd_64_samples.jsonl


In [ ]:
# Build the TB task output table for the sampled keys and write it to a JSONL file.
import json
from pathlib import Path

# Use the sampled primary keys already produced in the earlier sampling cell.
# These are the 64 keys we want to build records for.
if 'sampled_keys' not in globals():
    raise NameError("Run the sampling cell first so 'sampled_keys' is defined.")

# Locate the project root from the current notebook working directory.
base_dir = Path.cwd()
if not (base_dir / "HumanEval").exists():
    base_dir = next(
        (p for p in [base_dir, *base_dir.parents] if (p / "HumanEval").exists()),
        base_dir,
    )

# Paths to the four TB result files, one per approach.
files = {
    "Prompting": base_dir / "Prompting" / "Results" / "Qwen3-30B-A3B-Instruct-2507-TBtest-results.jsonl",
    "Finetuning": base_dir / "Finetune" / "Results" / "qwen3-30b-a3b-instruct-TBtest.jsonl",
    "RAFG": base_dir / "RAFG" / "Results" / "gemma-3-4b-it-TBtest.jsonl",
    "DRAFT": base_dir / "DRAFT" / "Results" / "qwen3-30b-a3b-instruct-TBtest.jsonl",
}

# Path to the ADR source data for the TB task.
source_path = base_dir / "Data" / "ADR-data" / "test.jsonl"


def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def get_primary_key(row):
    return row.get("PrimaryKey") or row.get("primary_key") or row.get("primaryKey")


def get_tb_value(row):
    # TB approach files hold the generated answer in Body, while some files may use Decision.
    return row.get("Body") or row.get("body")

# Load all rows from the four TB approaches.
approach_rows = {name: load_jsonl(path) for name, path in files.items()}

# Create a lookup from primary key to row for each approach.
lookups = {
    name: {get_primary_key(row): row for row in rows if get_primary_key(row) is not None}
    for name, rows in approach_rows.items()
}

# Load the ADR source rows and create a lookup by primary key.
source_rows = load_jsonl(source_path)
source_lookup = {
    get_primary_key(row): row
    for row in source_rows
    if get_primary_key(row) is not None
}

# Build one output dict per sampled primary key for the TB task.
output_rows = []
for pk in sampled_keys:
    source_row = source_lookup.get(pk)
    row_dict = {
        "PrimaryKey": pk,
        "Title": source_row.get("Title") if source_row else None,
        "Body": source_row.get("Body") if source_row else None,
    }
    for name in ["Prompting", "Finetuning", "RAFG", "DRAFT"]:
        row = lookups[name].get(pk)
        row_dict[name] = get_tb_value(row) if row is not None else None
    output_rows.append(row_dict)

# Save the combined records as JSONL in the HumanEval folder.
output_path = base_dir / "HumanEval" / "tb_64_samples.jsonl"
with output_path.open("w", encoding="utf-8") as f:
    for item in output_rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Wrote {len(output_rows)} records to {output_path}")

Wrote 64 records to d:\LAB\ADR_DRAFT\DRAFT\HumanEval\tb_64_samples.jsonl


## Architect study

In [ ]:
# Sample 14 additional keys from the 64-key pool and save them for the architect study.
import json
import random
from pathlib import Path

# Use the 64 sampled keys from the earlier cell.
if 'sampled_keys' not in globals():
    raise NameError("Run the sampling cell first so 'sampled_keys' is defined.")

base_dir = Path.cwd()
if not (base_dir / "HumanEval").exists():
    base_dir = next(
        (p for p in [base_dir, *base_dir.parents] if (p / "HumanEval").exists()),
        base_dir,
    )

random.seed(42)
sampled_14_keys = random.sample(sampled_keys, 14)

print("Sampled 14 primary keys from the 64:")
print(sampled_14_keys)

output_json_path = base_dir / "HumanEval" / "sampled_14_keys.json"
payload = {"Architect": sampled_14_keys}
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Saved 14 sampled keys to {output_json_path}")